# Colored Point-Cloud Generation — ShapeNet (airplane / car / chair)

Generates **colored point clouds** (XYZRGB) from ShapeNet meshes using **CloudCompare**.
This is the **naive-first** version (`docs/adr/0001`): we sample directly, with **no
dual-face fix**, so some clouds show salt-and-pepper colour **speckle** — that is
*expected*; fixing it is a later iteration.

**Pipeline:** install CloudCompare → read the 3 category zips from the shared
`Ortak/data/shapenetcore/` → inspect layout → extract 20 models/category → sample each to
100k XYZRGB points → validate (colour check + Plotly) → write outputs to `Ortak/colored_pc/`.

**Data lives in the shared Drive** (`docs/adr/0004`): the ShapeNet zips are stored in
`Ortak/data/shapenetcore/`, so **nobody re-downloads** — and Pelin needs **no HuggingFace
account**. The HF-download cell is kept only as an *optional* fallback.

**Prerequisites (each teammate, once):**
1. **Drive shortcut:** Drive (web) → *Shared with me* → right-click **Point Cloud Completion**
   → *Organize → Add shortcut to Drive* → *My Drive*. (Colab can't see "Shared with me" otherwise.)
2. **(Only if you ever run the optional HF download)** accept ShapeNetCore's license on HF and
   add `HF_TOKEN` to Colab *Secrets*. Not needed for the normal shared-data path.

**Scope / handoff (`docs/adr/0002`):** we output raw colored PLYs (native coordinates, 100k
points) + `manifest.csv` + `errors.log`. Normalization / FPS / partial-pair generation are
the teammate's downstream job.

> Outputs go into shared `Ortak/colored_pc/`, so both teammates see identical files. Because
> CloudCompare's sampling is random (no CLI seed), re-runs give *equivalent* clouds, not
> byte-identical ones — for identical data, read the PLYs already in `Ortak/`.


In [ ]:
# ============================================================
# Cell A — Configuration  (RUN ME FIRST)
# ------------------------------------------------------------
# Identical results whether EFE or PELIN runs it. Reads the ShapeNet zips from the
# SHARED Ortak folder (no re-download, no HuggingFace needed) and writes outputs back
# into Ortak so both of you see them.
#
# >>> ONE-TIME MANUAL STEP (each teammate, once) <<<
#   "Point Cloud Completion" is under "Shared with me", which Colab does NOT auto-mount.
#   Drive (web) -> Shared with me -> right-click "Point Cloud Completion"
#     -> Organize -> Add shortcut to Drive -> place in "My Drive"
#   After that, the SHARED_* paths below resolve for BOTH of you.
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')                      # mounts each user's My Drive (+ their shortcuts)

# --- Shared paths (identical for anyone who added the shortcut) ---
SHARED_ROOT     = "/content/drive/MyDrive/Point Cloud Completion"
ORTAK           = f"{SHARED_ROOT}/Ortak"                  # the common folder
SHARED_DATA_DIR = f"{ORTAK}/data/shapenetcore"           # the ShapeNet zips live here (we only READ them)
OUT_DIR         = f"{ORTAK}/colored_pc"                  # outputs land here -> both teammates see them

# --- Local scratch on the Colab VM (fast SSD, ephemeral; NOT Drive) ---
WORK_DIR = "/content/work"
RAW_DIR  = f"{WORK_DIR}/raw"                        # models extracted here from the shared zips
PLY_DIR  = f"{WORK_DIR}/colored_pc"                # PLYs written locally first, copied to OUT_DIR at end

# --- Categories -> ShapeNet synset (folder) IDs ---
CATEGORIES = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
SYNSETS = list(CATEGORIES.values())

# --- Knobs locked in planning ---
N_PER_CAT = 20          # models per category
N_POINTS  = 100_000     # points CloudCompare samples per model
SEED      = 42          # fixes WHICH models get picked -> same selection for both teammates

HF_REPO_ID = "ShapeNet/ShapeNetCore"   # only used by the OPTIONAL fallback download cell

# --- Setup local dirs, and fail LOUDLY if the shared shortcut isn't in place ---
for d in (WORK_DIR, RAW_DIR, PLY_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(ORTAK), (
    f"\n[STOP] Can't see the shared folder:\n  {ORTAK}\n"
    "Add the 'Point Cloud Completion' shortcut to your My Drive:\n"
    "Drive (web) -> Shared with me -> right-click -> Organize -> Add shortcut to Drive -> My Drive."
)
os.makedirs(OUT_DIR, exist_ok=True)
print("OK. Reading data from:", SHARED_DATA_DIR)
print("    Writing outputs to:", OUT_DIR)


In [ ]:
# ============================================================
# Cell B — Install CloudCompare + a virtual display, and Python deps
# ------------------------------------------------------------
# CloudCompare is a DESKTOP application (not a pip package): we install the binary
# onto the Colab VM with apt. It needs a display even in command-line mode, so we
# also install xvfb (X Virtual FrameBuffer) and will launch it via `xvfb-run`.
# This is the most fragile cell; the REAL functional test is Cell F (sampling one mesh).
# ============================================================
import shutil, subprocess

# System packages (Colab runs Ubuntu 22.04, whose 'universe' repo ships cloudcompare).
subprocess.run("apt-get update -qq", shell=True, check=True)
subprocess.run("apt-get install -y -qq cloudcompare xvfb", shell=True, check=True)

# Python deps not preinstalled on Colab (open3d reads/checks PLY colors).
subprocess.run("pip install -q -U huggingface_hub open3d", shell=True, check=True)

# Locate the binary (apt installs it as 'CloudCompare', capital C).
CC_BIN = shutil.which("CloudCompare") or shutil.which("cloudcompare")
assert CC_BIN, ("CloudCompare not found after apt install. If this Colab image lacks the "
                "package, we switch to a flatpak/AppImage install in this cell.")
print("CloudCompare:", CC_BIN, "| xvfb-run present:", bool(shutil.which("xvfb-run")))
print("Deps ready. Functional test is Cell F (sampling one mesh).")


In [ ]:
# ============================================================
# Cell C — Use the shared data in Ortak (NO download)
# ------------------------------------------------------------
# The 3 category zips already live in Ortak/data/shapenetcore/. We read them straight
# from the mounted shared folder (Cells D/E extract only the models we need to local).
# We DON'T copy whole zips and we DON'T re-download. This cell just hard-checks the zips
# are present, so a missing file / wrong shortcut fails loudly instead of silently
# re-downloading. If a zip really is missing, run the OPTIONAL HF cell just below.
# ============================================================
import os

missing = [s for s in SYNSETS if not os.path.exists(f"{SHARED_DATA_DIR}/{s}.zip")]
assert not missing, (
    f"\n[STOP] These zips are missing from {SHARED_DATA_DIR}:\n  "
    + ", ".join(f"{s}.zip" for s in missing)
    + "\nCheck the shared folder and your Drive shortcut, or run the OPTIONAL "
      "HF-download cell below to fetch them."
)
for s in SYNSETS:
    p = f"{SHARED_DATA_DIR}/{s}.zip"
    print(f"  {s}.zip: OK ({os.path.getsize(p) / 1e6:.0f} MB)")
print("All 3 category zips present in the shared folder — no download needed.")


In [ ]:
# ============================================================
# Cell C-ALT — OPTIONAL fallback: download the 3 zips from HuggingFace
# ------------------------------------------------------------
# DO NOT run this normally. It exists only to (re)populate Ortak/data/shapenetcore/
# if a zip is missing. It needs a HuggingFace account with the ShapeNetCore license
# accepted and HF_TOKEN in Colab Secrets — which is exactly the dependency we AVOID by
# using the shared data (Pelin doesn't have it). Downloads ONLY the 3 category zips,
# straight into the shared folder, so the normal Cell C then finds them.
# ============================================================
import os
from huggingface_hub import snapshot_download
from google.colab import userdata

os.makedirs(SHARED_DATA_DIR, exist_ok=True)
snapshot_download(
    repo_id=HF_REPO_ID, repo_type="dataset",
    local_dir=SHARED_DATA_DIR,
    allow_patterns=[f"{s}.zip" for s in SYNSETS],     # only the 3 categories we use
    token=userdata.get("HF_TOKEN"),
)
print("Downloaded the 3 category zips into:", SHARED_DATA_DIR)


In [ ]:
# ============================================================
# Cell D — Inspect the on-disk layout BEFORE writing sampling code
# ------------------------------------------------------------
# We assume nothing about v1 vs v2. Peek inside one shared zip (read straight from the
# Drive mount), extract ONE model, print its folder tree, and show the .mtl so we can
# SEE how textures are referenced (the OBJ -> MTL -> image "color chain"). If color goes
# missing later, it is because a link in this chain broke.
# ============================================================
import os, zipfile

probe_synset = SYNSETS[0]
probe_zip = f"{SHARED_DATA_DIR}/{probe_synset}.zip"      # read directly from the shared folder

with zipfile.ZipFile(probe_zip) as zf:
    names = zf.namelist()
    print(f"{probe_synset}.zip — {len(names)} entries. First 15:")
    for n in names[:15]:
        print("   ", n)

    obj_entries = [n for n in names if n.endswith("models/model_normalized.obj")]
    print(f"\nmodel_normalized.obj entries: {len(obj_entries)}")
    one_obj = obj_entries[0]
    model_dir = one_obj[: one_obj.index("models/")]      # "<synset>/<model_id>/"
    members = [n for n in names if n.startswith(model_dir)]
    zf.extractall(path=f"{WORK_DIR}/_probe", members=members)

probe_root = os.path.join(f"{WORK_DIR}/_probe", model_dir)
print(f"\nExtracted one model to: {probe_root}\nTree:")
for root, dirs, files in os.walk(probe_root):
    lvl = root.replace(probe_root, "").count(os.sep)
    print("   " * lvl, os.path.basename(root.rstrip("/")) + "/")
    for f in files:
        print("   " * (lvl + 1), f)

mtl = os.path.join(probe_root, "models", "model_normalized.mtl")
print("\n--- model_normalized.mtl (first 25 lines) ---")
if os.path.exists(mtl):
    with open(mtl) as fh:
        for i, line in zip(range(25), fh):
            print("   ", line.rstrip())
else:
    print("   (no .mtl at expected path — check the tree above)")


In [ ]:
# ============================================================
# Cell E — Extract N_PER_CAT models per category (DETERMINISTICALLY)
# ------------------------------------------------------------
# (A) We read each zip STRAIGHT from the mounted shared folder and extract only the
#     chosen models to local disk — never copying the whole multi-GB zip.
#     If this is ever slow/flaky over the Drive mount, fallback (B): copy the zip local
#     first (shutil.copy(f"{SHARED_DATA_DIR}/{synset}.zip", WORK_DIR)) then open that copy.
# We sort model IDs and take the first N so Efe and Pelin pick the SAME models. Each
# model's WHOLE folder (models/ + images/) is extracted intact, so the OBJ's relative
# texture paths resolve and CloudCompare can read color (see Cell D).
# ============================================================
import os, zipfile

selected = {}    # synset -> list of (model_id, model_dir_abspath)

for synset in SYNSETS:
    with zipfile.ZipFile(f"{SHARED_DATA_DIR}/{synset}.zip") as zf:   # read from shared folder
        names = zf.namelist()
        # model_id -> its model directory prefix ("<synset>/<model_id>/")
        dir_by_id = {}
        for n in names:
            if n.endswith("models/model_normalized.obj"):
                d = n[: n.index("models/")]
                dir_by_id[d.rstrip("/").split("/")[-1]] = d

        chosen = sorted(dir_by_id)[:N_PER_CAT]            # deterministic selection
        print(f"{synset}: {len(dir_by_id)} models -> extracting {len(chosen)}")

        roots = []
        for mid in chosen:
            d = dir_by_id[mid]
            zf.extractall(path=RAW_DIR, members=[n for n in names if n.startswith(d)])
            roots.append((mid, os.path.join(RAW_DIR, d)))
        selected[synset] = roots

print("\nExtracted per category:", {s: len(v) for s, v in selected.items()})


In [ ]:
# ============================================================
# Cell F — The CloudCompare sampling step (+ functional test on ONE model)
# ------------------------------------------------------------
# We keep EPFL's exact command and run it headless via xvfb. Flags:
#   -SILENT              no GUI / no prompts
#   -AUTO_SAVE OFF       don't save after every command
#   -C_EXPORT_FMT PLY    export clouds as PLY (stores per-point r,g,b)
#   -O <obj>             open the mesh
#   -SAMPLE_MESH POINTS N    scatter N color+geometry points over the surface
#   -SAVE_CLOUDS         write the result
# CloudCompare saves "<obj>_SAMPLED_POINTS.ply" NEXT TO the input; we move/rename it.
# ============================================================
import os, glob, shutil, subprocess
import numpy as np, open3d as o3d

def sample_colored_cloud(obj_path, out_ply, n_points=N_POINTS, cc_bin=CC_BIN):
    """Naive colored sampling (no dual-face fix -> speckle is expected)."""
    cmd = (f'xvfb-run -a {cc_bin} -SILENT -AUTO_SAVE OFF -C_EXPORT_FMT PLY '
           f'-O "{obj_path}" -SAMPLE_MESH POINTS {n_points} -SAVE_CLOUDS')
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    produced = [p for p in glob.glob(os.path.join(os.path.dirname(obj_path), "*.ply"))
                if "SAMPLED_POINTS" in os.path.basename(p)]
    if not produced:
        raise RuntimeError(f"No sampled PLY produced for {obj_path}")
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    shutil.move(produced[0], out_ply)
    return out_ply

# --- functional smoke test on one model (this exercises the Cell B install end-to-end) ---
ts = SYNSETS[0]
tid, troot = selected[ts][0]
tobj = os.path.join(troot, "models", "model_normalized.obj")
tout = sample_colored_cloud(tobj, f"{PLY_DIR}/{ts}/{tid}.ply")

pc = o3d.io.read_point_cloud(tout)
cols = np.asarray(pc.colors)
print(f"OK -> {tout}  ({os.path.getsize(tout) / 1e6:.1f} MB)")
print(f"points={len(pc.points)}  has_colors={pc.has_colors()}  color_std={cols.std():.4f}")
print("Color present." if pc.has_colors() and cols.std() > 0.01
      else "WARNING: near-constant/no color (textureless model or broken texture chain).")


In [ ]:
# ============================================================
# Cell G — Sample ALL selected models (the 60), robustly
# ------------------------------------------------------------
# Per-model try/except so one bad mesh can't abort the batch; failures -> errors list.
# We record a manifest row per success, incl. color_ok (the silent-failure guard:
# a valid-but-colorless PLY means the texture chain broke or the model is textureless).
# ============================================================
import os, traceback
import numpy as np, open3d as o3d
from tqdm import tqdm

NAME_BY_SYNSET = {v: k for k, v in CATEGORIES.items()}

def is_color_ok(ply_path, std_thresh=0.01):
    pc = o3d.io.read_point_cloud(ply_path)
    return bool(pc.has_colors() and np.asarray(pc.colors).std() > std_thresh)

manifest_rows, errors = [], []
jobs = [(s, mid, root) for s in SYNSETS for (mid, root) in selected[s]]

for synset, mid, root in tqdm(jobs, desc="Sampling 60 models"):
    obj = os.path.join(root, "models", "model_normalized.obj")
    out = f"{PLY_DIR}/{synset}/{mid}.ply"
    try:
        sample_colored_cloud(obj, out)
        manifest_rows.append({
            "synset": synset, "category": NAME_BY_SYNSET[synset], "model_id": mid,
            "source_obj": obj, "output_ply": out,
            "n_points": N_POINTS, "color_ok": is_color_ok(out),
        })
    except Exception:
        errors.append((synset, mid, traceback.format_exc()))

n_ok = sum(r["color_ok"] for r in manifest_rows)
print(f"\nSampled {len(manifest_rows)}/{len(jobs)} models; {len(errors)} failed.")
print(f"color_ok: {n_ok}/{len(manifest_rows)} "
      f"(colorless ones are likely textureless models, not bugs).")


In [ ]:
# ============================================================
# Cell H — Validate: see colors AND the dual-face speckle (Plotly, inline)
# ------------------------------------------------------------
# color_ok already flagged silent failures across all 60. Here we eyeball a few:
# Open3D's viewer can't run in Colab (no display), so we use a Plotly 3D scatter,
# downsampled for browser speed. Per ADR-0001, visible salt-and-pepper speckle is
# SUCCESS today (it is the problem we fix in a later iteration), not a failure.
# ============================================================
import numpy as np, open3d as o3d
import plotly.graph_objects as go

def show_cloud(ply_path, title, max_pts=25000):
    pc = o3d.io.read_point_cloud(ply_path)
    pts, cols = np.asarray(pc.points), np.asarray(pc.colors)
    if len(pts) > max_pts:                       # keep the browser smooth
        idx = np.random.choice(len(pts), max_pts, replace=False)
        pts, cols = pts[idx], (cols[idx] if len(cols) else cols)
    marker = dict(size=1.5)
    if len(cols):
        marker["color"] = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in cols]
    fig = go.Figure(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                                 mode="markers", marker=marker))
    fig.update_layout(title=title, height=500, scene_aspectmode="data",
                      margin=dict(l=0, r=0, t=30, b=0))
    fig.show()

# one example per category (first successful model)
for synset in SYNSETS:
    row = next((r for r in manifest_rows if r["synset"] == synset), None)
    if row:
        show_cloud(row["output_ply"],
                   f'{row["category"]} — {row["model_id"]} (color_ok={row["color_ok"]})')
    else:
        print(f"No successful model to show for {synset}.")


In [ ]:
# ============================================================
# Cell I — Hand off to the shared Ortak folder
# ------------------------------------------------------------
# Write manifest.csv + errors.log locally, then copy the whole colored_pc/ tree
# (60 small PLYs) from local disk to Ortak/colored_pc/ in ONE batch. This shared
# write IS the transfer: both teammates then see identical files (ADR-0002 / 0004).
# ============================================================
import os, shutil
import pandas as pd

pd.DataFrame(manifest_rows).to_csv(f"{PLY_DIR}/manifest.csv", index=False)
with open(f"{PLY_DIR}/errors.log", "w") as fh:
    if not errors:
        fh.write("No errors.\n")
    for synset, mid, tb in errors:
        fh.write(f"\n=== {synset}/{mid} ===\n{tb}\n")

# Local -> shared Drive (dirs_exist_ok so re-runs overwrite cleanly).
shutil.copytree(PLY_DIR, OUT_DIR, dirs_exist_ok=True)
print("Handoff complete ->", OUT_DIR)
for root, dirs, files in os.walk(OUT_DIR):
    lvl = root.replace(OUT_DIR, "").count(os.sep)
    print("   " * lvl, os.path.basename(root.rstrip("/")) + "/")
    for f in sorted(files)[:6]:
        print("   " * (lvl + 1), f)
